# Export

> Merge a LoRA adapter into the original Qwen3.5 checkpoint, convert it to GGUF for llama.cpp, and score the quantized files

In [ ]:
#| default_exp export

The Arduino UNO Q runs models with [llama.cpp](https://github.com/ggml-org/llama.cpp), which loads GGUF files. This notebook turns the fine-tuned Qwen3.5 0.8B from [Fine-tuning](06_finetune.ipynb), trained on 199 examples, into GGUF files at 8 and 4 bits, and scores each one on the test set through llama.cpp. The scores must match the MLX model's before a file goes to the board.

In [ ]:
#| export
import json, shutil, asyncio
from pathlib import Path
import httpx
from edgeslm.checks import check
from edgeslm.baseline import student_input, student_version
from edgeslm.finetune import SHORT_PROMPT

## Merging the adapter

mlx-lm's own GGUF export supports only Llama, Mistral and Mixtral models, so the GGUF file comes from llama.cpp's converter, `convert_hf_to_gguf.py`. The converter reads checkpoints in the Hugging Face layout. The mlx-community checkpoint we trained on is not in that layout: its tensor names start with `language_model.model.` instead of `model.language_model.`, its vision encoder is removed, and its convolution weights are transposed.

The adapter changes only linear projections, and the base weights of those 124 projections are identical in the two checkpoints. `fuse_lora` therefore merges the adapter into the original `Qwen/Qwen3.5-0.8B` checkpoint. It renames each adapted layer and adds `scale · Bᵀ·Aᵀ` to its weight, the formula mlx-lm uses when it fuses. The other files of the checkpoint are copied unchanged:

In [ ]:
#| export
def fuse_lora(
    base:str, # Directory of the original Hugging Face checkpoint
    adapters:str, # Directory of the mlx-lm adapters
    out:str, # Directory for the merged checkpoint
)->Path:
    "Merge mlx-lm LoRA adapters into a Hugging Face checkpoint"
    import torch  # imported here so the module loads without torch in CI
    from safetensors.torch import load_file, save_file
    base, adapters, out = Path(base), Path(adapters), Path(out)
    scale = json.loads((adapters/'adapter_config.json').read_text())['lora_parameters']['scale']
    lora = load_file(adapters/'adapters.safetensors')
    w = {}
    for fn in base.glob('*.safetensors'): w.update(load_file(fn))
    for t in sorted({k.rsplit('.lora_', 1)[0] for k in lora}):
        k = t.replace('language_model.model.', 'model.language_model.', 1) + '.weight'
        a, b = lora[t+'.lora_a'].float(), lora[t+'.lora_b'].float()
        w[k] = (w[k].float() + scale * b.T @ a.T).to(w[k].dtype)
    out.mkdir(parents=True, exist_ok=True)
    for f in base.iterdir():
        if f.is_file() and f.suffix != '.safetensors' and f.name != 'model.safetensors.index.json': shutil.copy(f, out/f.name)
    save_file(w, out/'model.safetensors', metadata={'format': 'pt'})
    return out

We checked the merge by loading the merged checkpoint with mlx-lm: it wrote the same messages as the adapter for the first three test records.

This cell merged the adapter and converted the result. It needs llama.cpp's command-line tools (`brew install llama.cpp`, version 0.8.0 here) and a checkout of llama.cpp at the same commit for `convert_hf_to_gguf.py`, and does not run in tests:

In [ ]:
#| eval: false
import subprocess, sys
from huggingface_hub import snapshot_download
LLAMA_CPP = Path('~/pro/dev/llama.cpp').expanduser()
run = Path('../models/qwen3.5-0.8b-n199')
hf = fuse_lora(snapshot_download('Qwen/Qwen3.5-0.8B'), run/'adapters', run/'hf')
gguf = run/'gguf'; gguf.mkdir(exist_ok=True)
bf16 = gguf/'qwen3.5-0.8b-edgeslm-bf16.gguf'
subprocess.run([sys.executable, LLAMA_CPP/'convert_hf_to_gguf.py', hf, '--outtype', 'bf16', '--outfile', bf16], env={'PYTHONPATH': str(LLAMA_CPP/'gguf-py')}, check=True)
for q in ('Q8_0', 'Q4_K_M'): subprocess.run(['llama-quantize', bf16, gguf/f'qwen3.5-0.8b-edgeslm-{q}.gguf', q], check=True)

## Scoring with llama.cpp

`score_gguf` sends each test record to a running `llama-server` and checks the message. It builds the prompt with `student_input`, the same chat-formatted string the MLX runs used, and decodes greedily. Any change in score therefore comes from llama.cpp and the quantization. Each row also keeps llama.cpp's token counts and timings:

In [ ]:
#| export
async def score_gguf(
    url:str, # Address of a running `llama-server`
    rows:list, # Test rows with `id`, `tuple` and `record`
    tok, # The model's tokenizer, to build the chat-formatted prompt
    name:str, # Name of the GGUF file, saved in each row
    n_par:int=4, # Requests in flight at once
    max_tokens:int=400, # Longest message to generate
)->list: # One row per input, with the message, its checks and llama.cpp's timings
    "Have a llama.cpp server write and check a message for each row"
    sem = asyncio.Semaphore(n_par)
    async with httpx.AsyncClient(timeout=300) as cli:
        async def one(r):
            async with sem:
                res = await cli.post(f'{url}/completion', json=dict(prompt=student_input(tok, r['record'], SHORT_PROMPT),
                    n_predict=max_tokens, temperature=0, top_k=1))
            j = res.json(); msg = j['content'].strip()
            return dict(id=r['id'], tuple=r['tuple'], message=msg, checks=check(r['record'], msg), model=name,
                        prompt=student_version(SHORT_PROMPT), decoding='greedy',
                        timings={k: j['timings'][k] for k in ('prompt_n', 'prompt_ms', 'predicted_n', 'predicted_ms')})
        return await asyncio.gather(*map(one, rows))

This cell scored each file, with `llama-server -m <file> --port 8091 -c 4096 -np 4` running in a terminal. It does not run in tests:

In [ ]:
#| eval: false
from transformers import AutoTokenizer
from edgeslm.dataset import read_jsonl, write_jsonl
test = read_jsonl('../data/test-v1.jsonl')
tok = AutoTokenizer.from_pretrained('../models/qwen3.5-0.8b-n199/hf')
q = 'Q4_K_M'  # restart the server with each file, then set q to match
write_jsonl(f'../data/gguf-v1-qwen3.5-0.8b-n199-{q}.jsonl', await score_gguf('http://localhost:8091', test, tok, f'qwen3.5-0.8b-edgeslm-{q}.gguf'))

## Results

In [ ]:
from edgeslm.dataset import read_jsonl
from edgeslm.baseline import score_table, markers

The MLX model in bf16 and the two GGUF files, scored on the same 270 test records:

In [ ]:
test = read_jsonl('../data/test-v1.jsonl')
runs = {'MLX bf16': read_jsonl('../data/finetune-v1-qwen3.5-0.8b-n199.jsonl'),
        **{f'GGUF {q}': read_jsonl(f'../data/gguf-v1-qwen3.5-0.8b-n199-{q}.jsonl') for q in ('Q8_0', 'Q4_K_M')}}
score_table(runs)

| Check | MLX bf16 | GGUF Q8_0 | GGUF Q4_K_M |
|---|---|---|---|
| all | 264 / 270 | 264 / 270 | 264 / 270 |
| format | 270 / 270 | 270 / 270 | 270 / 270 |
| coverage | 270 / 270 | 270 / 270 | 270 / 270 |
| no_invented | 270 / 270 | 270 / 270 | 270 / 270 |
| severity | 270 / 270 | 270 / 270 | 270 / 270 |
| no_numbers | 270 / 270 | 270 / 270 | 270 / 270 |
| length | 264 / 270 | 264 / 270 | 264 / 270 |
| amount | 270 / 270 | 270 / 270 | 270 / 270 |

Quantization costs no checks. The changed messages we read differ in wording only, such as "no longer sees its leaves" for "has not seen its leaves". The number of messages identical to the MLX model's shows how many changed:

In [ ]:
{k: sum(a['message'] == b['message'] for a, b in zip(v, runs['MLX bf16'])) for k, v in runs.items()}

{'MLX bf16': 270, 'GGUF Q8_0': 265, 'GGUF Q4_K_M': 251}

The 4-bit file also writes the right marker for every item:

In [ ]:
markers(runs['GGUF Q4_K_M'], test)

Counter({('early', '👀'): 209, ('moderate', '⚠️'): 148, ('severe', '🚨'): 112})

The 4-bit file is 542 MB, against 834 MB for 8 bits and 1.56 GB for bf16. It scores as well as the others, so it is the file for the UNO Q. Speed and memory on the board are still to be measured.

## Running on the UNO Q

The colleague with the board needs llama.cpp built on it, the 4-bit file `qwen3.5-0.8b-edgeslm-Q4_K_M.gguf`, and the folder `data/uno-q` from this repo. The folder holds three test prompts, `prompt-N.txt`, already in the model's chat format, and the 4-bit file's message for each, `expected-N.txt`. This cell wrote them, and does not run in tests:

In [ ]:
#| eval: false
q4 = read_jsonl('../data/gguf-v1-qwen3.5-0.8b-n199-Q4_K_M.jsonl')
for i in range(3):
    Path(f'../data/uno-q/prompt-{i}.txt').write_text(student_input(tok, test[i]['record'], SHORT_PROMPT))
    Path(f'../data/uno-q/expected-{i}.txt').write_text(q4[i]['message'])

The steps below run on the board. First, build llama.cpp at the version used here (0.8.0) on the board's Debian:

```sh
sudo apt install git cmake build-essential time
git clone https://github.com/ggml-org/llama.cpp && cd llama.cpp
git checkout 8642541b2
cmake -B build && cmake --build build --config Release -j 4
```

The tools are in `build/bin`. Copy the GGUF file and the `uno-q` folder next to them, then run each step from `build/bin`.

**Write a message.** This writes the message for the first test prompt with greedy decoding, as the scoring did:

```sh
./llama-completion -m qwen3.5-0.8b-edgeslm-Q4_K_M.gguf -f uno-q/prompt-0.txt -n 400 --temp 0 --top-k 1 -no-cnv --no-display-prompt
```

The output ends with `[end of text]`. Compare the text before it with `uno-q/expected-0.txt`. On a Mac it is identical. The board computes on a different processor, so a few words can change. A different marker, location or number of paragraphs is a real problem: report it.

**Measure speed.** Test prompts average 168 tokens and messages 71, so benchmark those sizes on the four cores:

```sh
./llama-bench -m qwen3.5-0.8b-edgeslm-Q4_K_M.gguf -p 168 -n 71 -t 4
```

The table gives tokens per second for reading the prompt (`pp168`) and for writing the message (`tg71`). The time for one message is about 168 ÷ pp168 + 71 ÷ tg71 seconds.

**Measure memory.** Run the first step under GNU `time`:

```sh
/usr/bin/time -v ./llama-completion -m qwen3.5-0.8b-edgeslm-Q4_K_M.gguf -f uno-q/prompt-0.txt -n 400 --temp 0 --top-k 1 -no-cnv --no-display-prompt
```

The line `Maximum resident set size` gives the peak memory in kilobytes.

Send back the three outputs, the `llama-bench` table and the peak memory.

### Scoring all test records on the board

The scope requires final scores measured on the target board. If a computer with this repo can reach the board over the network, the colleague starts a server on the board:

```sh
./llama-server -m qwen3.5-0.8b-edgeslm-Q4_K_M.gguf --host 0.0.0.0 --port 8091 -c 4096
```

On the computer, run the scoring cell above with the board's address, `http://<board-ip>:8091`, and `n_par=1`, so that each timing measures one message on its own. Each row then holds the board's timings for that message. We have not run this yet.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()